# 06 — Strategy Comparison
Compares all 5 strategies (FedAvg, TrimmedMean, Krum, RobustFL, SSFG) at 30% attacker ratio.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))
from src.configs.paths import RESULTS_DIR, ARTIFACTS_DIR

strategies = {
    'FedAvg (unprotected)': 'round_results_fedavg_ratio_30pct.csv',
    'Trimmed Mean':         'round_results_trimmed_mean_ratio_30pct.csv',
    'Krum':                 'round_results_krum_ratio_30pct.csv',
    'RobustFL (Ours)':      'round_results_robust_ratio_30pct.csv',
    'SSFG (Ours)':          'round_results_ssfg_ratio_30pct.csv',
}

dfs = {}
for label, fname in strategies.items():
    path = RESULTS_DIR / fname
    if path.exists():
        dfs[label] = pd.read_csv(path)
        print(f'Loaded: {label}')
    else:
        print(f'MISSING: {fname}')

In [ ]:
# Macro F1 comparison: all strategies on one chart
palette = ['tomato', 'orange', 'gold', 'steelblue', 'mediumseagreen']
fig, ax = plt.subplots(figsize=(13, 5))
for (label, df), color in zip(dfs.items(), palette):
    ax.plot(df['round'], df['macro_f1'], marker='o', markersize=3, label=label, color=color)
ax.axvline(10.5, color='gray', linestyle='--', alpha=0.6, label='Attack start')
ax.set_xlabel('FL Round')
ax.set_ylabel('Macro F1-Score')
ax.set_title('Strategy Comparison — Macro F1 (30% Attacker Ratio, Label Flip)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'comparison_macro_f1.png', dpi=150)
plt.show()

In [ ]:
# FPR comparison
fig, ax = plt.subplots(figsize=(13, 5))
for (label, df), color in zip(dfs.items(), palette):
    ax.plot(df['round'], df['fpr'], marker='s', markersize=3, label=label, color=color)
ax.axvline(10.5, color='gray', linestyle='--', alpha=0.6)
ax.set_xlabel('FL Round')
ax.set_ylabel('False Positive Rate')
ax.set_title('Strategy Comparison — FPR (30% Attacker Ratio)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'plots' / 'comparison_fpr.png', dpi=150)
plt.show()

In [ ]:
# Final round bar chart comparison
if dfs:
    rows = []
    for label, df in dfs.items():
        last = df.iloc[-1]
        rows.append({'Strategy': label, 'Macro F1': last['macro_f1'],
                     'Accuracy': last['accuracy'], 'FPR': last['fpr']})
    summary = pd.DataFrame(rows).set_index('Strategy')

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for ax, col, color in zip(axes, ['Macro F1', 'Accuracy', 'FPR'], ['steelblue', 'green', 'orange']):
        summary[col].plot(kind='bar', ax=ax, color=color, edgecolor='white')
        ax.set_title(col)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=30)
    plt.suptitle('Final Round Metrics — Strategy Comparison (30% Attackers)', fontsize=13)
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / 'plots' / 'comparison_final_bar.png', dpi=150)
    plt.show()
    print(summary.round(4).to_string())